# Paper-faithful LIFE on ISOT (human fake-vs-real) — negative control

Runs the **actual LIFE method, paper-faithful**, on **ISOT** (Fake.csv / True.csv), which is
**human-written on both sides**. Faithful = **LLaMA-2-7B reconstruction model** (not GPT-2) +
the **sigmoid + BCE head** (paper Eq 11–12), i.e. `train_bce.py` + `model_bce.py` — NOT the
BMES/CRF head.

This is a bigger, independent, different-domain replication of **§7j** (LLaMA-2-7B + BCE, HF-vs-HR
on PolitiFact++, where the faithful BCE head sat **at the majority prior** — LIFE detects
LLM-*generation*, not fakeness). Expected here: BCE **Acc ≈ majority baseline, low Macro-F1**.

**Labels:** Fake → `human_fake`, True → `human_true`. `train_bce.py` derives the binary label from
the suffix (`endswith('_fake')` → fake=1, real=0), so it needs **no edits** for ISOT.

**⚠️ Read the result carefully.** ISOT is **not source-matched**: ~99% of True.csv is Reuters
newswire, ~0% of Fake.csv is. So a **near-prior / low-F1** result cleanly supports *"LIFE can't
recognize human-written fake news"*; an **above-prior** result is **ambiguous** (LLaMA perplexity
may read newswire style, not veracity). The clean, topic-matched control remains HF-vs-HR (§7j).

**➡️ Source-strip toggle (this run).** The un-stripped run scored **Acc 75.06 ± 2.17 / Macro-F1
75.04 ± 2.18** — clearly above the 50% baseline, i.e. the classifier IS separating the classes,
likely via the Reuters artifact. Set **`STRIP_SOURCE = True`** (paths cell) to remove the
`<LOCATION> (Reuters) -` datelines + `(Reuters)` tokens and re-run. **Read the delta:** if Acc
falls toward ~50% → the source tell drove the 75% (supports the hypothesis); if it stays well
above 50% → LLaMA is still reading residual Reuters-vs-tabloid *register* (still not veracity —
the clean answer stays §7j). The stripped run writes to `isot_life_stripped/` so the baseline
artifacts are preserved for comparison; `STRIP_SOURCE = False` reproduces the baseline.

**Scale:** 1,000 articles/class. **GPU required** — stage 1 (BERT) + stage 3 (LLaMA-2-7B, bf16
≈14 GB → T4/L4/A100). **5 stages:** 0 convert → 1 key sentences (BERT) → 2 merge → 3 LLaMA
perplexity features → 4 train the BCE head (+ multi-seed mean±std).

In [ ]:
!pip install -q transformers datasets nltk tqdm sentencepiece accelerate  # torch preinstalled

In [ ]:
import torch, nltk
nltk.download('punkt', quiet=True); nltk.download('punkt_tab', quiet=True)
print('CUDA available:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE - set Runtime to GPU')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

STRIP_SOURCE = True    # True = strip Reuters datelines/locations (the follow-up run);
                       # False = reproduce the 75.06 +/- 2.17 un-stripped baseline
SUFFIX     = '_stripped' if STRIP_SOURCE else ''
STRIP_FLAG = '--strip_source' if STRIP_SOURCE else ''

PROJECT_DIR = '/content/drive/MyDrive/LIFE'
ISOT_DIR    = f'{PROJECT_DIR}/dataset/data/ISOT'          # holds Fake.csv / True.csv
WORK        = f'{ISOT_DIR}/isot_life{SUFFIX}'             # stripped run -> own dir (baseline kept)
CONVERTED   = f'{WORK}/converted'                         # stage 0 out: ISOT_fake/true.jsonl
FEATURES    = f'{WORK}/features_llama'                    # stage 3 out: LLaMA feature jsonl
KEYSENTS    = f'{WORK}/keysents.jsonl'                    # stage 1 out
BERT_CKPT   = f'{WORK}/isot_bert_model.pt'                # stage 1 key-sentence extractor
TRAIN_JSONL = f'{WORK}/train.jsonl'                       # stage 4 split (kept OUT of FEATURES
TEST_JSONL  = f'{WORK}/test.jsonl'                        #   so it is not re-ingested as input)

os.chdir(PROJECT_DIR)  # so `dataset/...` and `LIFE_train/...` script paths + imports resolve
os.makedirs(WORK, exist_ok=True)
print('ISOT Fake.csv found:', os.path.isfile(f'{ISOT_DIR}/Fake.csv'))
print('ISOT True.csv found:', os.path.isfile(f'{ISOT_DIR}/True.csv'))

## Stage 0 — convert ISOT CSV → JSONL, subsample 1,000 / class
Fake → `human_fake`, True → `human_true`; synthesizes ids (ISOT has none). `{STRIP_FLAG}` is
`--strip_source` when `STRIP_SOURCE = True` (removes Reuters datelines/`(Reuters)`), else empty.
Seed-42 sampling is fixed, so stripped and baseline runs use the **same 1,000/class articles**.

In [ ]:
!python dataset/0_convert_isot.py --input_dir "{ISOT_DIR}" --output_dir "{CONVERTED}" --sample_size 1000 --seed 42 {STRIP_FLAG}

## Stage 1 — key-sentence extraction (BERT)
Trains `bert-base-uncased` to separate fake/true, then leave-one-sentence-out to pick the top-10
sentences per article (~10–20 min on a T4 for 2k articles). *On ISOT the BERT separates the
classes easily via the Reuters artifact — but it only **selects which sentences get scored**;
LIFE's verdict comes from the LLaMA perplexity features + BCE head in stages 3–4.*

In [ ]:
!python dataset/1_keySentenceExtraction.py --data_dir "{CONVERTED}" --output_file "{KEYSENTS}" --top_k 10 --model_path "{BERT_CKPT}" --gpu 0

## Stage 2 — merge the key sentences back into the article JSONL
In-place: adds a `sentence` field to `CONVERTED/ISOT_fake.jsonl` and `ISOT_true.jsonl`.

In [ ]:
!python dataset/2_concate.py --folder_path "{CONVERTED}" --important_sentences_file "{KEYSENTS}"

## Stage 3 — LLaMA-2-7B perplexity features (paper-faithful reconstruction model)
The paper's reconstruction mLLM is **LLaMA-2-7B** (§4.1.3), scored with the SentencePiece
`--scorer llama` path in `backend_utils.SPLlamaTokenizerPPLCalc`. `NousResearch/Llama-2-7b-hf`
is an **ungated mirror** of the official weights → **no HF token needed** (swap to
`meta-llama/Llama-2-7b-hf` if you have Meta approval). bf16 ≈14 GB; ~15–40 min for 2k articles.
First run downloads ~13 GB of weights.

In [ ]:
!python dataset/3_gen_features_local.py --input_dir "{CONVERTED}" --output_dir "{FEATURES}" --model NousResearch/Llama-2-7b-hf --scorer llama --dtype bfloat16 --gpu 0

## Stage 4 — train the paper BCE head (sigmoid + binary cross-entropy)
`train_bce.py` / `model_bce.py`: masked mean-pool → one sigmoid probability per article, BCE loss
(fake=1, real=0), evaluated at article level — no BMES/CRF/majority-vote. `--split_dataset` uses
the deterministic seed-0 train/test split. **Watch Acc vs the majority baseline and Macro-F1:**
at-prior + low F1 → LIFE has no fingerprint for human-written fake news (the point of this run).
`real = human_true`, `fake = human_fake`.

In [ ]:
!python LIFE_train/train_bce.py --split_dataset --data_path "{FEATURES}" --train_path "{TRAIN_JSONL}" --test_path "{TEST_JSONL}" --num_train_epochs 50 --seed 0

### Stage 4b — multi-seed mean ± std (the honest read for a negative control)
The split is fixed at seed 0; only model init + batch order vary. §7j showed single seeds are
misleading here — some collapse to "always real", a few catch a handful of fakes — so report
**mean ± std** over seeds. Features are tiny → this is a couple of minutes. Extend `range(10)` to
`range(21)` to match the paper's 0–20 sweep.

In [ ]:
import subprocess, re, numpy as np
accs, f1s = [], []
for seed in range(10):
    out = subprocess.run(
        ['python', 'LIFE_train/train_bce.py', '--split_dataset',
         '--data_path', FEATURES, '--train_path', TRAIN_JSONL, '--test_path', TEST_JSONL,
         '--num_train_epochs', '50', '--seed', str(seed)],
        capture_output=True, text=True).stdout
    a = re.findall(r'Accuracy: ([\d.]+)', out)          # printed every epoch; take the last
    f = re.findall(r'Macro F1 Score: ([\d.]+)', out)
    if a and f:
        accs.append(float(a[-1])); f1s.append(float(f[-1]))
        print(f'seed {seed:>2}: Acc {a[-1]:>5}  Macro-F1 {f[-1]:>5}')
    else:
        print(f'seed {seed:>2}: FAILED (no metrics parsed)')
print(f'\nBCE head over {len(accs)} seeds:  '
      f'Acc {np.mean(accs):.2f} ± {np.std(accs):.2f}   '
      f'Macro-F1 {np.mean(f1s):.2f} ± {np.std(f1s):.2f}')

## Notes
- **Baseline vs. stripped (the A/B this run is for):** un-stripped = **75.06 ± 2.17 / 75.04 ± 2.18**
  (clear above-chance separation). Re-run with `STRIP_SOURCE = True` and compare: a large drop toward
  ~50% attributes the 75% to the Reuters source tell (supports "LIFE can't recognize human fakeness");
  a small drop means residual register/style is still being read (still not veracity — the clean
  control stays §7j). Same 1,000/class articles both ways (seed 42), so the delta is purely the
  removed source tells.
- **What collapse would look like:** BCE Acc ≈ majority baseline (~50% at 1:1) with low Macro-F1 and a
  tight Acc std / noisy F1 std (the prior-collapse signature from §7j) → the fingerprint can't
  separate human fake from human real.
- **Sync to Drive before running:** `dataset/0_convert_isot.py` (new). Everything else is reused
  UNCHANGED and should already be on Drive from your MF-vs-MR / §7j runs:
  `dataset/1_keySentenceExtraction.py`, `2_concate.py`, `3_gen_features_local.py`, `backend_utils.py`,
  `LIFE_train/train_bce.py`, `LIFE_train/model_bce.py`, `LIFE_train/dataloader.py`, `model.py`.
- **1:1 class balance:** this run uses 1,000 fake / 1,000 real, so the majority baseline is ~50%
  (cleaner to read than §7j's 2:1 HF/HR, where the prior was ~67%).
- **Change scale:** edit `--sample_size` in Stage 0 (0 = all ~45k). **Re-run cleanly:** delete `WORK`
  first; Stage 1 *loads* `BERT_CKPT` if it exists instead of retraining.
- **Compare vs the released BMES/CRF head** (the §7j A-side): run `LIFE_train/train.py` on the same
  `FEATURES` — but note that head is nearly insensitive to feature quality (§7h/§7i), so its number
  is a mechanism artifact, not a fingerprint signal. The BCE head is the honest instrument.